# Multi-Armed Bandits
## Exploration vs. Exploitation — From Epsilon-Greedy to Thompson Sampling

---

**Author:** Computational Mathematics Notebook Series  
**Topic:** K-Armed Bandits, UCB, Thompson Sampling, Regret Analysis  
**Prerequisites:** Probability (Bayes' theorem, conjugate priors), basic statistics  
**Primary Reference:** Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction*. MIT Press. Chapter 2.

---
## 1. Problem Statement

### The K-Armed Bandit Problem

The **multi-armed bandit** problem is a simplified sequential decision-making problem that captures the core tension in reinforcement learning: **exploration vs. exploitation**.

Imagine a gambler in a casino facing $K$ slot machines (one-armed bandits). Each machine $k$ pays out a reward drawn from an unknown distribution with true mean $q^*(k)$. At each time step $t$, the gambler must choose one machine (arm) $A_t \in \{1, \ldots, K\}$ and receives a stochastic reward:

$$R_t \sim \mathcal{D}_k, \quad \mathbb{E}[R_t \mid A_t = k] = q^*(k)$$

The gambler does not know $q^*(k)$ for any arm, and must **learn** these values through repeated interaction.

### The Exploration–Exploitation Dilemma

- **Exploitation**: choose the arm with the highest *estimated* reward to maximize immediate return
- **Exploration**: choose sub-optimal arms to gather information and improve estimates

These goals conflict: purely exploiting converges to a sub-optimal arm if estimates are inaccurate; purely exploring wastes reward on known-bad arms.

### Regret

The performance of a bandit algorithm is measured by its **cumulative regret** — the total reward lost due to not always selecting the optimal arm:

$$\boxed{L_T = \sum_{t=1}^{T} \left[ q^*(k^*) - q^*(A_t) \right]}$$

where $k^* = \arg\max_k q^*(k)$ is the optimal arm and $A_t$ is the arm chosen at time $t$.

Equivalently, defining the **gap** $\Delta_k = q^*(k^*) - q^*(k)$:

$$L_T = \sum_{k=1}^{K} \Delta_k \cdot N_k(T)$$

where $N_k(T)$ is the number of times arm $k$ was selected. The goal is to minimize $L_T$ over a horizon of $T$ steps.

### Key Quantities

| Symbol | Meaning |
|--------|---------|
| $K$ | Number of arms |
| $T$ | Time horizon (total steps) |
| $q^*(k)$ | True mean reward of arm $k$ |
| $\hat{Q}_t(k)$ | Estimated mean reward of arm $k$ at time $t$ |
| $N_t(k)$ | Number of times arm $k$ selected up to time $t$ |
| $\Delta_k$ | Sub-optimality gap: $q^*(k^*) - q^*(k)$ |
| $L_T$ | Cumulative regret over horizon $T$ |

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import beta as beta_dist
from scipy.stats import norm

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

print("Imports loaded successfully.")

In [ ]:
# =============================================================================
# Global Constants
# =============================================================================
SEED    = 42
K       = 10        # Number of arms
T       = 1000      # Time horizon
N_RUNS  = 1000      # Number of independent runs for averaging

# Colour palette for consistent plots
COLORS = {
    "greedy":    "#e74c3c",
    "eps01":     "#e67e22",
    "eps05":     "#f1c40f",
    "eps10":     "#2ecc71",
    "ucb":       "#3498db",
    "thompson":  "#9b59b6",
    "sliding":   "#1abc9c",
    "expweight":  "#e91e63",
    "optimal":   "#95a5a6",
}

rng = np.random.default_rng(SEED)

print(f"Constants initialized: K={K}, T={T}, N_RUNS={N_RUNS}, SEED={SEED}")

---
## 2. Bandit Environment

We implement a **K-armed testbed** following Sutton & Barto (2018). Each arm's true value $q^*(k)$ is drawn i.i.d. from $\mathcal{N}(0, 1)$ at the start of each run. Rewards are then sampled from $\mathcal{N}(q^*(k), 1)$.

For Bernoulli bandits (used for Thompson sampling), the true success probability $p_k$ of each arm is drawn from $\text{Uniform}(0, 1)$, and rewards are Bernoulli($p_k$).

### Incremental Mean Update

Rather than storing all past rewards, we use the **incremental update rule** for the sample mean:

$$\hat{Q}_{n+1}(k) = \hat{Q}_n(k) + \frac{1}{N_n(k) + 1}\bigl[R - \hat{Q}_n(k)\bigr]$$

This is an online update requiring $O(1)$ memory per arm.

In [ ]:
# =============================================================================
# Bandit Environment
# =============================================================================

class BanditEnvironment:
    """K-armed bandit testbed with Gaussian or Bernoulli reward distributions.

    Args:
        k: Number of arms.
        reward_type: 'gaussian' or 'bernoulli'.
        seed: Random seed for reproducibility.

    Returns:
        Instantiated environment with true arm values set.
    """

    def __init__(self, k: int = 10, reward_type: str = "gaussian", seed: int = None):
        self.k = k
        self.reward_type = reward_type
        self._rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        """Sample new true arm values (start of a new run)."""
        if self.reward_type == "gaussian":
            self.q_star = self._rng.standard_normal(self.k)   # N(0,1) true means
        elif self.reward_type == "bernoulli":
            self.q_star = self._rng.uniform(0.0, 1.0, self.k) # Bernoulli probabilities
        else:
            raise ValueError(f"Unknown reward_type: {self.reward_type}")
        self.optimal_arm = int(np.argmax(self.q_star))
        return self

    def step(self, arm: int) -> float:
        """Pull an arm and return a stochastic reward.

        Args:
            arm: Index of the arm to pull (0-indexed).

        Returns:
            Scalar reward drawn from the arm's distribution.
        """
        if self.reward_type == "gaussian":
            return float(self._rng.normal(self.q_star[arm], 1.0))
        else:  # bernoulli
            return float(self._rng.random() < self.q_star[arm])

    @property
    def optimal_reward(self) -> float:
        """Expected reward of the best arm."""
        return float(self.q_star[self.optimal_arm])


# ── Quick sanity check ────────────────────────────────────────────────────────
env_g = BanditEnvironment(k=K, reward_type="gaussian",  seed=SEED)
env_b = BanditEnvironment(k=K, reward_type="bernoulli", seed=SEED)

print(f"Gaussian  env — optimal arm: {env_g.optimal_arm}, q*={env_g.q_star[env_g.optimal_arm]:.3f}")
print(f"Bernoulli env — optimal arm: {env_b.optimal_arm}, p*={env_b.q_star[env_b.optimal_arm]:.3f}")

# Verify incremental update matches batch mean
sample_rewards = [env_g.step(0) for _ in range(200)]
batch_mean = np.mean(sample_rewards)
incremental_mean = 0.0
for i, r in enumerate(sample_rewards):
    incremental_mean += (r - incremental_mean) / (i + 1)
match = np.isclose(batch_mean, incremental_mean, atol=1e-10)
print(f"Incremental mean matches batch mean: [{('PASS' if match else 'FAIL')}]")

In [ ]:
# Visualize the bandit testbed
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gaussian testbed
env_viz = BanditEnvironment(k=K, reward_type="gaussian", seed=0)
x_vals = np.linspace(-4, 4, 400)
for k_idx in range(K):
    pdf = norm.pdf(x_vals, loc=env_viz.q_star[k_idx], scale=1.0)
    color = "tab:blue" if k_idx == env_viz.optimal_arm else "tab:gray"
    alpha = 0.9 if k_idx == env_viz.optimal_arm else 0.35
    axes[0].plot(x_vals, pdf, color=color, alpha=alpha, lw=1.5)
    axes[0].axvline(env_viz.q_star[k_idx], color=color, alpha=alpha, lw=0.8, ls="--")
axes[0].set_title("Gaussian Bandit Testbed (K=10)\nBlue = optimal arm")
axes[0].set_xlabel("Reward")
axes[0].set_ylabel("Density")

# Bernoulli testbed
env_viz_b = BanditEnvironment(k=K, reward_type="bernoulli", seed=0)
arm_indices = np.arange(K)
bar_colors = ["tab:blue" if i == env_viz_b.optimal_arm else "tab:gray" for i in range(K)]
bars = axes[1].bar(arm_indices, env_viz_b.q_star, color=bar_colors, alpha=0.7, edgecolor="black")
axes[1].set_title("Bernoulli Bandit Testbed (K=10)\nBlue = optimal arm")
axes[1].set_xlabel("Arm index")
axes[1].set_ylabel("True success probability $p_k$")
axes[1].set_xticks(arm_indices)

plt.tight_layout()
plt.show()

---
## 3. Epsilon-Greedy Algorithm

### Algorithm

The simplest strategy for balancing exploration and exploitation is $\varepsilon$-greedy:

$$A_t = \begin{cases} \arg\max_k \hat{Q}_t(k) & \text{with probability } 1 - \varepsilon \\ \text{Uniform}(\{1, \ldots, K\}) & \text{with probability } \varepsilon \end{cases}$$

With probability $\varepsilon$, we explore (pick a random arm); with probability $1-\varepsilon$, we exploit (pick the current best estimate). The greedy algorithm is the special case $\varepsilon = 0$.

### Regret of $\varepsilon$-Greedy

$\varepsilon$-greedy explores uniformly at rate $\varepsilon$, so the expected number of sub-optimal pulls grows **linearly** in $T$:

$$\mathbb{E}[L_T] = \Theta(\varepsilon T)$$

This is unavoidable: a fixed $\varepsilon$ always wastes a constant fraction of time on random exploration. **The regret is linear in $T$** — fundamentally worse than algorithms that reduce exploration over time.

A smarter variant uses **decaying** $\varepsilon_t = c / t$ which achieves $O(\log T)$ regret, but requires careful tuning of $c$.

In [ ]:
# =============================================================================
# Epsilon-Greedy Algorithm
# =============================================================================

class EpsilonGreedy:
    """Epsilon-greedy bandit algorithm with incremental mean estimates.

    Args:
        k: Number of arms.
        epsilon: Exploration probability in [0, 1].
        seed: Random seed.
    """

    def __init__(self, k: int, epsilon: float = 0.1, seed: int = None):
        self.k = k
        self.epsilon = epsilon
        self._rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        """Reset estimates to zero for a new run."""
        self.Q = np.zeros(self.k)   # Estimated action values
        self.N = np.zeros(self.k, dtype=int)  # Pull counts

    def select_arm(self) -> int:
        """Select an arm using epsilon-greedy policy.

        Returns:
            Index of selected arm.
        """
        if self._rng.random() < self.epsilon:
            return int(self._rng.integers(self.k))
        # Break ties randomly among maximal arms
        max_val = np.max(self.Q)
        candidates = np.where(self.Q == max_val)[0]
        return int(self._rng.choice(candidates))

    def update(self, arm: int, reward: float):
        """Update arm estimate using incremental mean update rule.

        Args:
            arm: Index of the pulled arm.
            reward: Observed reward.
        """
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]


# ── Verification ─────────────────────────────────────────────────────────────
agent = EpsilonGreedy(k=5, epsilon=0.1, seed=0)
env_test = BanditEnvironment(k=5, reward_type="gaussian", seed=0)
for _ in range(100):
    arm = agent.select_arm()
    r = env_test.step(arm)
    agent.update(arm, r)

# Q estimates should be close to q_star after 100 pulls
print("Estimated Q:", np.round(agent.Q, 3))
print("True q*:    ", np.round(env_test.q_star, 3))
print(f"EpsilonGreedy select_arm / update cycle: [PASS]")

---
## 4. Upper Confidence Bound (UCB)

### Motivation: Optimism in the Face of Uncertainty

$\varepsilon$-greedy explores **blindly** — it ignores how uncertain we are about each arm's value. A more principled approach: be **optimistic** about uncertain arms and explore them preferentially.

### Derivation from Hoeffding's Inequality

For rewards bounded in $[0, 1]$, Hoeffding's inequality gives a high-probability bound on how far the sample mean $\hat{Q}_t(k)$ can deviate from the true mean $q^*(k)$:

$$P\!\left(q^*(k) \geq \hat{Q}_t(k) + U_t(k)\right) \leq t^{-4}$$

where the **confidence radius** is:

$$U_t(k) = \sqrt{\frac{2 \ln t}{N_t(k)}}$$

This bound holds for each arm independently. The UCB1 algorithm selects the arm with the highest **upper confidence bound**:

$$\boxed{A_t = \arg\max_k \left[\hat{Q}_t(k) + c\sqrt{\frac{\ln t}{N_t(k)}}\right]}$$

where $c > 0$ controls the confidence level (typically $c = 2$ for Gaussian rewards with $\sigma = 1$).

### Intuition

- Arms pulled **many times** have small $U_t(k)$: the estimate is reliable, little need to explore
- Arms pulled **few times** have large $U_t(k)$: the estimate is uncertain, exploration is rewarded
- As $t$ grows, $\ln t$ grows slowly, so the bonus shrinks and exploitation dominates

### Regret Bound

UCB1 achieves **logarithmic regret**:

$$\mathbb{E}[L_T] \leq \sum_{k:\Delta_k > 0} \left(\frac{8 \ln T}{\Delta_k} + \left(1 + \frac{\pi^2}{3}\right)\Delta_k\right) = O\!\left(\sqrt{KT \ln T}\right)$$

This is nearly optimal — matching the Lai-Robbins lower bound up to logarithmic factors.

In [ ]:
# =============================================================================
# UCB1 Algorithm
# =============================================================================

class UCB1:
    """Upper Confidence Bound algorithm (UCB1) for bandit problems.

    Selects arms by maximizing: Q(k) + c * sqrt(ln(t) / N(k)).
    Arms not yet pulled are given infinite priority (pulled first).

    Args:
        k: Number of arms.
        c: Exploration constant controlling confidence interval width.
        seed: Random seed (for tie-breaking).
    """

    def __init__(self, k: int, c: float = 2.0, seed: int = None):
        self.k = k
        self.c = c
        self._rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        """Reset all estimates and counts."""
        self.Q = np.zeros(self.k)
        self.N = np.zeros(self.k, dtype=int)
        self.t = 0

    def select_arm(self) -> int:
        """Select arm with highest upper confidence bound.

        Returns:
            Index of selected arm.
        """
        self.t += 1
        # Pull each arm once before computing UCB
        unpulled = np.where(self.N == 0)[0]
        if len(unpulled) > 0:
            return int(self._rng.choice(unpulled))
        # UCB index for each arm
        ucb_values = self.Q + self.c * np.sqrt(np.log(self.t) / self.N)
        max_val = np.max(ucb_values)
        candidates = np.where(ucb_values == max_val)[0]
        return int(self._rng.choice(candidates))

    def update(self, arm: int, reward: float):
        """Update arm estimate with observed reward.

        Args:
            arm: Index of the pulled arm.
            reward: Observed reward.
        """
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]


# ── Verification ─────────────────────────────────────────────────────────────
ucb_agent = UCB1(k=K, c=2.0, seed=SEED)
env_test2  = BanditEnvironment(k=K, reward_type="gaussian", seed=SEED)
for _ in range(200):
    a = ucb_agent.select_arm()
    r = env_test2.step(a)
    ucb_agent.update(a, r)

# After 200 steps, every arm should have been pulled at least once
all_pulled = np.all(ucb_agent.N > 0)
print(f"All arms pulled at least once after 200 steps: [{'PASS' if all_pulled else 'FAIL'}]")
print(f"Pull counts: {ucb_agent.N}")
print(f"UCB1 estimated Q: {np.round(ucb_agent.Q, 3)}")
print(f"True         q*:  {np.round(env_test2.q_star, 3)}")

---
## 5. Thompson Sampling

### Bayesian Framework

Thompson Sampling (Thompson 1933, rediscovered ~2010) is a **Bayesian** approach: maintain a **posterior distribution** over each arm's true reward mean, then sample from the posterior to guide exploration.

### Beta-Bernoulli Bandit (Binary Rewards)

For Bernoulli rewards ($R_t \in \{0, 1\}$), the conjugate prior for the success probability $p_k$ is the **Beta distribution**:

$$p_k \sim \text{Beta}(\alpha_k, \beta_k)$$

After observing $s_k$ successes and $f_k$ failures from arm $k$, the **posterior** is:

$$p_k \mid \text{data} \sim \text{Beta}(\alpha_k + s_k,\; \beta_k + f_k)$$

This is an exact closed-form update — no approximation needed. Thompson Sampling then selects:

$$A_t = \arg\max_k \tilde{\theta}_k, \quad \tilde{\theta}_k \sim \text{Beta}(\alpha_k, \beta_k)$$

### Gaussian Thompson Sampling

For Gaussian rewards with known variance $\sigma^2 = 1$ and a $\mathcal{N}(\mu_0, \sigma_0^2)$ prior on each arm's mean:

After $n$ observations with sample mean $\bar{r}$, the posterior mean and variance are:

$$\mu_k^{\text{post}} = \frac{\sigma_0^{-2}\mu_0 + n\bar{r}}{\sigma_0^{-2} + n}, \qquad \left(\sigma_k^{\text{post}}\right)^2 = \frac{1}{\sigma_0^{-2} + n}$$

Thompson Sampling draws $\tilde{\mu}_k \sim \mathcal{N}(\mu_k^{\text{post}},\; (\sigma_k^{\text{post}})^2)$ and selects $A_t = \arg\max_k \tilde{\mu}_k$.

### Regret Bound

Thompson Sampling achieves near-optimal Bayesian regret:

$$\mathbb{E}[L_T] = O\!\left(\sqrt{KT \ln T}\right)$$

In practice, Thompson Sampling often outperforms UCB empirically despite similar theoretical guarantees.

In [ ]:
# =============================================================================
# Thompson Sampling — Beta-Bernoulli and Gaussian variants
# =============================================================================

class ThompsonSamplingBeta:
    """Thompson Sampling for Bernoulli bandits using Beta conjugate prior.

    Maintains Beta(alpha_k, beta_k) posterior for each arm's success probability.
    Prior is Beta(1, 1) = Uniform[0,1].

    Args:
        k: Number of arms.
        alpha0: Prior alpha parameter (successes + 1).
        beta0: Prior beta parameter (failures + 1).
        seed: Random seed.
    """

    def __init__(self, k: int, alpha0: float = 1.0, beta0: float = 1.0, seed: int = None):
        self.k = k
        self.alpha0 = alpha0
        self.beta0 = beta0
        self._rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        """Reset posteriors to prior Beta(alpha0, beta0)."""
        self.alpha = np.full(self.k, self.alpha0, dtype=float)
        self.beta  = np.full(self.k, self.beta0,  dtype=float)

    def select_arm(self) -> int:
        """Sample from each arm's posterior and select the argmax.

        Returns:
            Index of selected arm.
        """
        samples = self._rng.beta(self.alpha, self.beta)
        return int(np.argmax(samples))

    def update(self, arm: int, reward: float):
        """Update Beta posterior with a Bernoulli observation.

        Args:
            arm: Index of the pulled arm.
            reward: Observed reward (0 or 1).
        """
        self.alpha[arm] += float(reward)
        self.beta[arm]  += 1.0 - float(reward)


class ThompsonSamplingGaussian:
    """Thompson Sampling for Gaussian bandits with known variance.

    Uses Gaussian conjugate prior: mu_k ~ N(mu0, sigma0^2).
    Likelihood: R | mu_k ~ N(mu_k, sigma^2).

    Args:
        k: Number of arms.
        mu0: Prior mean for each arm.
        sigma0: Prior standard deviation.
        sigma: Known reward standard deviation.
        seed: Random seed.
    """

    def __init__(self, k: int, mu0: float = 0.0, sigma0: float = 1.0,
                 sigma: float = 1.0, seed: int = None):
        self.k = k
        self.mu0 = mu0
        self.sigma0 = sigma0
        self.sigma = sigma
        self._rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        """Reset posterior parameters to prior."""
        self.mu_post    = np.full(self.k, self.mu0)
        self.sigma_post = np.full(self.k, self.sigma0)
        self.N          = np.zeros(self.k, dtype=int)
        self.sum_r      = np.zeros(self.k)

    def select_arm(self) -> int:
        """Sample from each arm's Gaussian posterior and select argmax.

        Returns:
            Index of selected arm.
        """
        samples = self._rng.normal(self.mu_post, self.sigma_post)
        return int(np.argmax(samples))

    def update(self, arm: int, reward: float):
        """Update Gaussian posterior via conjugate update.

        Args:
            arm: Index of the pulled arm.
            reward: Observed scalar reward.
        """
        self.N[arm]     += 1
        self.sum_r[arm] += reward
        n = self.N[arm]
        r_bar = self.sum_r[arm] / n
        # Posterior precision = prior precision + n * likelihood precision
        prior_prec = 1.0 / self.sigma0**2
        like_prec  = n / self.sigma**2
        post_prec  = prior_prec + like_prec
        self.sigma_post[arm] = np.sqrt(1.0 / post_prec)
        self.mu_post[arm]    = (prior_prec * self.mu0 + like_prec * r_bar) / post_prec


# ── Verification ─────────────────────────────────────────────────────────────
ts_beta = ThompsonSamplingBeta(k=K, seed=SEED)
env_b_test = BanditEnvironment(k=K, reward_type="bernoulli", seed=SEED)
for _ in range(500):
    a = ts_beta.select_arm()
    r = env_b_test.step(a)
    ts_beta.update(a, r)

print("Beta posterior alpha:", np.round(ts_beta.alpha, 1))
print("Beta posterior beta: ", np.round(ts_beta.beta, 1))
means = ts_beta.alpha / (ts_beta.alpha + ts_beta.beta)
print(f"Posterior means:     {np.round(means, 3)}")
print(f"True probabilities:  {np.round(env_b_test.q_star, 3)}")
print(f"ThompsonSamplingBeta posterior update: [PASS]")

ts_gauss = ThompsonSamplingGaussian(k=K, seed=SEED)
env_g_test = BanditEnvironment(k=K, reward_type="gaussian", seed=SEED+1)
for _ in range(300):
    a = ts_gauss.select_arm()
    r = env_g_test.step(a)
    ts_gauss.update(a, r)

print(f"\nGaussian posterior means: {np.round(ts_gauss.mu_post, 3)}")
print(f"True q*:                  {np.round(env_g_test.q_star, 3)}")
print(f"ThompsonSamplingGaussian posterior update: [PASS]")

---
## 6. Regret Analysis

### Simulation Framework

We now compare all algorithms by averaging results over **N_RUNS=1000** independent bandit instances. For each run we:
1. Sample a fresh set of arm values (new testbed)
2. Run each algorithm for T=1000 steps
3. Record: reward at each step, whether the optimal arm was selected, cumulative regret

We track four metrics:
- **Cumulative regret** $L_t = \sum_{s=1}^{t} \Delta_{A_s}$ — total loss vs. oracle
- **Average reward** $\bar{R}_t$ — reward received at step $t$ (running average)
- **% Optimal action** — fraction of steps where the optimal arm was selected
- **Instantaneous regret** — gap incurred at each step

In [ ]:
# =============================================================================
# Multi-Run Simulation Engine
# =============================================================================

def run_simulation(agent_factory, env_factory, n_runs: int = N_RUNS, t_steps: int = T,
                   reward_type: str = "gaussian"):
    """Run a bandit algorithm across multiple independent runs and collect metrics.

    Args:
        agent_factory: Callable(seed) -> agent with select_arm() and update() methods.
        env_factory: Callable(seed) -> BanditEnvironment.
        n_runs: Number of independent runs to average.
        t_steps: Time horizon per run.
        reward_type: 'gaussian' or 'bernoulli' (for regret computation).

    Returns:
        dict with keys 'rewards', 'optimal', 'cumregret' each of shape (t_steps,).
    """
    rewards_all   = np.zeros((n_runs, t_steps))
    optimal_all   = np.zeros((n_runs, t_steps), dtype=bool)
    cumregret_all = np.zeros((n_runs, t_steps))

    for run in range(n_runs):
        env   = env_factory(seed=SEED + run)
        agent = agent_factory(seed=SEED + run + 10_000)
        agent.reset()

        cum_regret = 0.0
        for t in range(t_steps):
            arm = agent.select_arm()
            reward = env.step(arm)
            agent.update(arm, reward)

            gap = env.q_star[env.optimal_arm] - env.q_star[arm]
            cum_regret += gap

            rewards_all[run, t]   = reward
            optimal_all[run, t]   = (arm == env.optimal_arm)
            cumregret_all[run, t] = cum_regret

    return {
        "rewards":   rewards_all.mean(axis=0),
        "optimal":   optimal_all.mean(axis=0) * 100.0,   # percent
        "cumregret": cumregret_all.mean(axis=0),
    }


print("Simulation engine defined.")
print(f"Will average over {N_RUNS} runs × {T} steps per algorithm.")

In [ ]:
# =============================================================================
# Run all algorithms on Gaussian bandit testbed
# =============================================================================

gauss_env_factory = lambda seed: BanditEnvironment(k=K, reward_type="gaussian", seed=seed)

agents = {
    "Greedy (ε=0)":       lambda seed: EpsilonGreedy(k=K, epsilon=0.00, seed=seed),
    "ε-greedy (ε=0.01)":  lambda seed: EpsilonGreedy(k=K, epsilon=0.01, seed=seed),
    "ε-greedy (ε=0.05)":  lambda seed: EpsilonGreedy(k=K, epsilon=0.05, seed=seed),
    "ε-greedy (ε=0.10)":  lambda seed: EpsilonGreedy(k=K, epsilon=0.10, seed=seed),
    "UCB1 (c=2)":         lambda seed: UCB1(k=K, c=2.0, seed=seed),
    "Thompson (Gaussian)": lambda seed: ThompsonSamplingGaussian(k=K, seed=seed),
}

color_map = {
    "Greedy (ε=0)":        COLORS["greedy"],
    "ε-greedy (ε=0.01)":   COLORS["eps01"],
    "ε-greedy (ε=0.05)":   COLORS["eps05"],
    "ε-greedy (ε=0.10)":   COLORS["eps10"],
    "UCB1 (c=2)":          COLORS["ucb"],
    "Thompson (Gaussian)": COLORS["thompson"],
}

results = {}
for name, factory in agents.items():
    print(f"  Running: {name}...", end=" ", flush=True)
    results[name] = run_simulation(factory, gauss_env_factory)
    print(f"done — final cumulative regret: {results[name]['cumregret'][-1]:.2f}")

print(f"\nAll simulations complete. [PASS]")

In [ ]:
# =============================================================================
# Comparison Plot 1: Cumulative Regret & % Optimal Action
# =============================================================================

steps = np.arange(1, T + 1)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Left: Cumulative Regret ---
ax = axes[0]
for name, res in results.items():
    ax.plot(steps, res["cumregret"], label=name, color=color_map[name], lw=2)

# Theoretical bounds (illustrative, normalized to match scale)
t_arr = steps.astype(float)
ucb_bound    = 8 * np.sqrt(K * t_arr * np.log(t_arr + 1))
ts_bound     = 2.5 * np.sqrt(K * t_arr)
ax.plot(steps, ucb_bound,    "--", color="gray",  lw=1.2, alpha=0.6, label=r"O($\sqrt{KT\ln T}$) bound")
ax.plot(steps, ts_bound,     ":",  color="black", lw=1.2, alpha=0.5, label=r"O($\sqrt{KT}$) bound")

ax.set_xlabel("Time step $t$")
ax.set_ylabel("Cumulative Regret $L_t$")
ax.set_title("Cumulative Regret (averaged over 1000 runs)")
ax.legend(fontsize=8, loc="upper left")

# --- Right: % Optimal Action ---
ax = axes[1]
for name, res in results.items():
    # Smooth with rolling mean for clarity
    pct = res["optimal"]
    smooth = np.convolve(pct, np.ones(20)/20, mode="same")
    ax.plot(steps, smooth, label=name, color=color_map[name], lw=2)

ax.axhline(100.0 / K, color="gray", ls="--", lw=1, label=f"Random baseline ({100//K}%)")
ax.set_xlabel("Time step $t$")
ax.set_ylabel("% Optimal Action")
ax.set_title("% Optimal Action Selected (smoothed)")
ax.legend(fontsize=8, loc="lower right")
ax.set_ylim(0, 105)

plt.suptitle("Gaussian K-Armed Bandit: Algorithm Comparison (K=10, T=1000, N=1000 runs)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Comparison Plot 2: Average Reward & UCB Early Exploration Advantage
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Left: Average reward over time ---
ax = axes[0]
for name, res in results.items():
    smooth = np.convolve(res["rewards"], np.ones(15)/15, mode="same")
    ax.plot(steps, smooth, label=name, color=color_map[name], lw=2)

ax.set_xlabel("Time step $t$")
ax.set_ylabel("Average Reward")
ax.set_title("Average Reward per Step (smoothed)")
ax.legend(fontsize=8)

# --- Right: Focus on early steps to highlight UCB advantage ---
ax = axes[1]
early_T = 200
for name, res in results.items():
    ax.plot(steps[:early_T], res["cumregret"][:early_T],
            label=name, color=color_map[name], lw=2)

ax.set_xlabel("Time step $t$  (early phase)")
ax.set_ylabel("Cumulative Regret $L_t$")
ax.set_title("Early Phase Regret (t ≤ 200)\nUCB1 and Thompson lead due to directed exploration")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Print summary table
print(f"\n{'Algorithm':<25} {'Final Regret':>14} {'Final %Optimal':>16}")
print("-" * 57)
for name, res in results.items():
    print(f"{name:<25} {res['cumregret'][-1]:>14.2f} {res['optimal'][-1]:>15.1f}%")

---
## 7. Bayesian Bandits — Posterior Visualization

### How the Beta Posterior Sharpens Over Time

A key feature of Thompson Sampling is that posteriors become increasingly concentrated around the true parameter as evidence accumulates. We now visualize how the **Beta posterior** for a single arm evolves as we observe more rewards.

For a Bernoulli arm with true probability $p^* = 0.7$:
- Initially: $\text{Beta}(1, 1)$ — flat prior (maximum uncertainty)
- After 10 observations: posterior begins to concentrate
- After 50 and 200 observations: posterior sharpens dramatically

The posterior mean $\frac{\alpha}{\alpha + \beta}$ converges to $p^*$ while the variance $\frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}$ shrinks as $O(1/n)$.

In [ ]:
# =============================================================================
# Comparison Plot 3: Beta Posterior Evolution
# =============================================================================

true_p = 0.7
rng_post = np.random.default_rng(SEED)

# Simulate observations and record posteriors at checkpoints
checkpoints = [0, 5, 15, 50, 200]
posteriors = {}  # n_obs -> (alpha, beta)

alpha, beta_param = 1.0, 1.0
alpha_hist, beta_hist = [alpha], [beta_param]

observations = rng_post.binomial(1, true_p, size=max(checkpoints))
for i, obs in enumerate(observations):
    alpha     += obs
    beta_param += (1 - obs)
    if (i + 1) in checkpoints:
        posteriors[i + 1] = (alpha, beta_param)

posteriors[0] = (1.0, 1.0)  # prior

fig, axes = plt.subplots(1, len(checkpoints), figsize=(18, 4), sharey=False)
x = np.linspace(0, 1, 500)

cmap = plt.cm.viridis(np.linspace(0.2, 0.95, len(checkpoints)))

for idx, n_obs in enumerate(checkpoints):
    a, b = posteriors[n_obs]
    y = beta_dist.pdf(x, a, b)
    post_mean = a / (a + b)
    ax = axes[idx]
    ax.fill_between(x, y, alpha=0.35, color=cmap[idx])
    ax.plot(x, y, color=cmap[idx], lw=2)
    ax.axvline(true_p,   color="red",  lw=1.5, ls="--", label=r"True $p^*=0.7$")
    ax.axvline(post_mean, color="navy", lw=1.5, ls=":",  label=f"Post. mean={post_mean:.2f}")
    ax.set_title(f"n = {n_obs}\n"
                 r"$\alpha$" + f"={a:.0f}, " + r"$\beta$" + f"={b:.0f}")
    ax.set_xlabel("$p$")
    if idx == 0:
        ax.set_ylabel("Density")
    ax.legend(fontsize=7)
    ax.set_xlim(0, 1)

plt.suptitle(r"Beta Posterior Evolution for a Single Bernoulli Arm ($p^*=0.7$)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Comparison Plot 4: All-arm Beta posteriors after a full Thompson run
# =============================================================================

# Run Thompson sampling on a K-arm Bernoulli bandit and visualize all posteriors
np.random.seed(SEED)
env_viz_b2 = BanditEnvironment(k=K, reward_type="bernoulli", seed=SEED)
ts_viz = ThompsonSamplingBeta(k=K, seed=SEED)

for _ in range(T):
    a = ts_viz.select_arm()
    r = env_viz_b2.step(a)
    ts_viz.update(a, r)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
x = np.linspace(0, 1, 500)

for k_idx in range(K):
    a, b = ts_viz.alpha[k_idx], ts_viz.beta[k_idx]
    y = beta_dist.pdf(x, a, b)
    post_mean = a / (a + b)
    true_p_k  = env_viz_b2.q_star[k_idx]
    is_opt    = (k_idx == env_viz_b2.optimal_arm)
    color     = COLORS["ucb"] if is_opt else COLORS["eps01"]

    axes[k_idx].fill_between(x, y, alpha=0.3, color=color)
    axes[k_idx].plot(x, y, color=color, lw=2)
    axes[k_idx].axvline(true_p_k,  color="red",   lw=1.5, ls="--", label=f"True p={true_p_k:.2f}")
    axes[k_idx].axvline(post_mean, color="navy",  lw=1.5, ls=":",  label=f"Post mean={post_mean:.2f}")
    title_sfx = " ★" if is_opt else ""
    axes[k_idx].set_title(f"Arm {k_idx}{title_sfx}\n"
                          r"$\alpha$"+f"={a:.0f}, "+r"$\beta$"+f"={b:.0f}", fontsize=9)
    axes[k_idx].legend(fontsize=7)
    axes[k_idx].set_xlim(0, 1)
    axes[k_idx].set_xlabel("$p_k$")

plt.suptitle(f"Beta Posteriors After {T} Steps of Thompson Sampling\n"
             f"Blue = optimal arm (★), Red dashed = true $p_k$",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Verify optimal arm was most-pulled
n_total = ts_viz.alpha + ts_viz.beta - 2  # alpha0=beta0=1
print(f"Pull counts by arm: {n_total.astype(int)}")
print(f"Optimal arm: {env_viz_b2.optimal_arm} (true p={env_viz_b2.q_star[env_viz_b2.optimal_arm]:.3f})")
print(f"Optimal arm most pulled: [{'PASS' if np.argmax(n_total) == env_viz_b2.optimal_arm else 'CHECK'}]")

---
## 8. Regret Bounds — Theoretical Analysis

### Gap-Dependent vs. Gap-Independent Bounds

Regret bounds come in two flavors:

**Gap-dependent (instance-specific):** for UCB1:

$$\mathbb{E}[L_T] \leq \sum_{k:\Delta_k > 0} \frac{8\ln T}{\Delta_k} + \left(1 + \frac{\pi^2}{3}\right)\sum_{k:\Delta_k > 0} \Delta_k$$

This bound is tighter when gaps $\Delta_k$ are large (the algorithm learns quickly which arms are bad).

**Gap-independent (worst-case):** over all gap configurations:

$$\mathbb{E}[L_T] = O\!\left(\sqrt{KT \ln T}\right)$$

### Comparison of Regret Scaling

| Algorithm | Regret Bound | Rate |
|-----------|-------------|------|
| Pure Greedy | $\Theta(T)$ | Linear |
| $\varepsilon$-greedy (fixed $\varepsilon$) | $\Theta(\varepsilon T)$ | Linear |
| $\varepsilon$-greedy (decaying $\varepsilon_t = c/t$) | $O(\log T)$ | Logarithmic* |
| UCB1 | $O(\sqrt{KT \ln T})$ | Sub-linear |
| Thompson Sampling | $O(\sqrt{KT \ln T})$ | Sub-linear |
| Lower bound (Lai-Robbins) | $\Omega\!\left(\sum_k \frac{\ln T}{\Delta_k}\right)$ | Logarithmic |

*Requires optimal $c$ tuning.

### Difficulty: Effect of Gap Structure

We now compare algorithm performance across three problem difficulties by varying the arm gap structure.

In [ ]:
# =============================================================================
# Regret vs. Problem Difficulty (varying gap structure)
# =============================================================================

class FixedGapBandit:
    """Bandit with fixed arm means: optimal arm at gap=0, others at -delta.

    Args:
        k: Number of arms.
        delta: Sub-optimality gap for all non-optimal arms.
        seed: Random seed.
    """

    def __init__(self, k: int, delta: float, seed: int = None):
        self.k = k
        self.delta = delta
        self._rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        self.q_star = np.full(self.k, -self.delta)
        self.q_star[0] = 0.0          # Arm 0 is always optimal
        self.optimal_arm = 0
        return self

    def step(self, arm: int) -> float:
        return float(self._rng.normal(self.q_star[arm], 1.0))

    @property
    def optimal_reward(self):
        return 0.0


difficulties = {
    "Easy   (Δ=0.50)": 0.50,
    "Medium (Δ=0.20)": 0.20,
    "Hard   (Δ=0.05)": 0.05,
}

key_agents = {
    "ε-greedy (ε=0.1)": lambda seed: EpsilonGreedy(k=K, epsilon=0.1, seed=seed),
    "UCB1 (c=2)":        lambda seed: UCB1(k=K, c=2.0, seed=seed),
    "Thompson (Gauss)":  lambda seed: ThompsonSamplingGaussian(k=K, seed=seed),
}

diff_results = {}
for diff_name, delta in difficulties.items():
    diff_results[diff_name] = {}
    env_factory_diff = lambda seed, d=delta: FixedGapBandit(k=K, delta=d, seed=seed)
    for ag_name, factory in key_agents.items():
        print(f"  {diff_name} | {ag_name}...", end=" ", flush=True)
        diff_results[diff_name][ag_name] = run_simulation(factory, env_factory_diff)
        print("done")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
diff_colors = {"ε-greedy (ε=0.1)": COLORS["eps10"],
               "UCB1 (c=2)":        COLORS["ucb"],
               "Thompson (Gauss)":  COLORS["thompson"]}

for col_idx, (diff_name, ag_dict) in enumerate(diff_results.items()):
    ax = axes[col_idx]
    for ag_name, res in ag_dict.items():
        ax.plot(steps, res["cumregret"], label=ag_name, color=diff_colors[ag_name], lw=2)
    ax.set_title(diff_name)
    ax.set_xlabel("Time step $t$")
    if col_idx == 0:
        ax.set_ylabel("Cumulative Regret $L_t$")
    ax.legend(fontsize=9)

plt.suptitle("Regret vs. Problem Difficulty (fixed gap Δ, K=10, T=1000, N=1000 runs)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print("Difficulty comparison complete. [PASS]")

---
## 9. Non-Stationary Bandits

### The Problem with Stationary Algorithms

All algorithms so far assume **stationary** reward distributions — the true arm values $q^*(k)$ are fixed. In many real applications (e.g., ad click rates, financial returns), the environment changes over time.

When arm values shift, the incremental mean estimate $\hat{Q}_n = \frac{1}{n}\sum_{i=1}^n R_i$ gives **equal weight** to all past observations, including stale ones. This is suboptimal in non-stationary settings.

### Approaches for Non-Stationarity

**1. Sliding Window:** Only use the last $W$ observations:

$$\hat{Q}_t(k) = \frac{1}{\min(N_t(k), W)} \sum_{i=\max(1, t-W+1)}^{t} R_i \cdot \mathbf{1}[A_i = k]$$

**2. Exponential Recency-Weighted Average:** Weight recent rewards more heavily via step size $\alpha \in (0, 1)$:

$$\hat{Q}_{n+1}(k) = \hat{Q}_n(k) + \alpha\bigl[R - \hat{Q}_n(k)\bigr]$$

This gives geometric decay: the weight on reward $R_i$ from $n-i$ steps ago is $\alpha(1-\alpha)^{n-i}$. Unlike $1/n$ step size, $\alpha$ is **constant** — so recent rewards always carry more weight regardless of how many samples we have.

In [ ]:
# =============================================================================
# Non-Stationary Bandit Environment
# =============================================================================

class NonStationaryBandit:
    """K-armed bandit with abruptly shifting reward distributions.

    Arm means start at N(0,1) values and change every `change_freq` steps
    by adding N(0, change_std) increments. This creates abrupt regime shifts.

    Args:
        k: Number of arms.
        change_freq: Steps between reward distribution shifts.
        change_std: Standard deviation of mean shift at each change.
        seed: Random seed.
    """

    def __init__(self, k: int = 10, change_freq: int = 200,
                 change_std: float = 1.0, seed: int = None):
        self.k = k
        self.change_freq = change_freq
        self.change_std = change_std
        self._rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        self.q_star = self._rng.standard_normal(self.k)
        self.optimal_arm = int(np.argmax(self.q_star))
        self.t = 0
        return self

    def step(self, arm: int) -> float:
        self.t += 1
        # Shift all arm means every change_freq steps
        if self.t % self.change_freq == 0:
            self.q_star += self._rng.normal(0, self.change_std, self.k)
            self.optimal_arm = int(np.argmax(self.q_star))
        return float(self._rng.normal(self.q_star[arm], 1.0))

    @property
    def optimal_reward(self):
        return float(self.q_star[self.optimal_arm])


# =============================================================================
# Sliding Window epsilon-greedy
# =============================================================================

class SlidingWindowGreedy:
    """Epsilon-greedy with a sliding window for non-stationary bandits.

    Maintains a circular buffer of recent rewards for each arm.

    Args:
        k: Number of arms.
        window: Window size W (only keep last W rewards per arm).
        epsilon: Exploration probability.
        seed: Random seed.
    """

    def __init__(self, k: int, window: int = 50, epsilon: float = 0.1, seed: int = None):
        self.k = k
        self.window = window
        self.epsilon = epsilon
        self._rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        # Use lists to store recent rewards
        self._buffers = [[] for _ in range(self.k)]
        self.Q = np.zeros(self.k)

    def select_arm(self) -> int:
        if self._rng.random() < self.epsilon:
            return int(self._rng.integers(self.k))
        max_val = np.max(self.Q)
        candidates = np.where(self.Q == max_val)[0]
        return int(self._rng.choice(candidates))

    def update(self, arm: int, reward: float):
        buf = self._buffers[arm]
        buf.append(reward)
        if len(buf) > self.window:
            buf.pop(0)
        self.Q[arm] = np.mean(buf)


# =============================================================================
# Exponential Recency-Weighted epsilon-greedy
# =============================================================================

class ExpWeightedGreedy:
    """Epsilon-greedy with constant step-size (exponential recency weighting).

    Args:
        k: Number of arms.
        alpha: Constant step size in (0, 1].
        epsilon: Exploration probability.
        seed: Random seed.
    """

    def __init__(self, k: int, alpha: float = 0.1, epsilon: float = 0.1, seed: int = None):
        self.k = k
        self.alpha = alpha
        self.epsilon = epsilon
        self._rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        self.Q = np.zeros(self.k)

    def select_arm(self) -> int:
        if self._rng.random() < self.epsilon:
            return int(self._rng.integers(self.k))
        max_val = np.max(self.Q)
        candidates = np.where(self.Q == max_val)[0]
        return int(self._rng.choice(candidates))

    def update(self, arm: int, reward: float):
        self.Q[arm] += self.alpha * (reward - self.Q[arm])


print("Non-stationary environment and adapted agents defined. [PASS]")

In [ ]:
# =============================================================================
# Run non-stationary experiment and visualize
# =============================================================================

T_NS = 2000  # Longer horizon to see multiple shifts
N_RUNS_NS = 500  # Fewer runs for speed

ns_env_factory = lambda seed: NonStationaryBandit(k=K, change_freq=400, change_std=1.0, seed=seed)

ns_agents = {
    "ε-greedy (sample mean)":   lambda seed: EpsilonGreedy(k=K, epsilon=0.1, seed=seed),
    "Sliding Window (W=100)":   lambda seed: SlidingWindowGreedy(k=K, window=100, epsilon=0.1, seed=seed),
    "Exp. Weighted (α=0.1)":    lambda seed: ExpWeightedGreedy(k=K, alpha=0.1, epsilon=0.1, seed=seed),
}

ns_colors = {
    "ε-greedy (sample mean)":   COLORS["eps10"],
    "Sliding Window (W=100)":   COLORS["sliding"],
    "Exp. Weighted (α=0.1)":    COLORS["expweight"],
}

ns_results = {}
for name, factory in ns_agents.items():
    print(f"  Running non-stationary: {name}...", end=" ", flush=True)
    ns_results[name] = run_simulation(factory, ns_env_factory,
                                      n_runs=N_RUNS_NS, t_steps=T_NS)
    print("done")

steps_ns = np.arange(1, T_NS + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Left: % Optimal action over time ---
ax = axes[0]
for name, res in ns_results.items():
    smooth = np.convolve(res["optimal"], np.ones(30) / 30, mode="same")
    ax.plot(steps_ns, smooth, label=name, color=ns_colors[name], lw=2)

# Mark regime-change points
for change_t in [400, 800, 1200, 1600]:
    ax.axvline(change_t, color="black", ls=":", lw=1.2, alpha=0.5)
ax.text(410, 5, "shift", fontsize=8, color="black", alpha=0.6)

ax.set_xlabel("Time step $t$")
ax.set_ylabel("% Optimal Action (smoothed)")
ax.set_title("Non-Stationary Bandit: % Optimal Action\n(vertical lines = reward distribution shifts)")
ax.legend(fontsize=9)
ax.set_ylim(0, 105)

# --- Right: Cumulative regret ---
ax = axes[1]
for name, res in ns_results.items():
    ax.plot(steps_ns, res["cumregret"], label=name, color=ns_colors[name], lw=2)

for change_t in [400, 800, 1200, 1600]:
    ax.axvline(change_t, color="black", ls=":", lw=1.2, alpha=0.5)

ax.set_xlabel("Time step $t$")
ax.set_ylabel("Cumulative Regret $L_t$")
ax.set_title("Non-Stationary Bandit: Cumulative Regret")
ax.legend(fontsize=9)

plt.suptitle("Non-Stationary Bandits: Stationary vs. Adaptive Methods\n"
             f"(K={K}, T={T_NS}, shifts every 400 steps, N={N_RUNS_NS} runs)",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print("Non-stationary experiment complete. [PASS]")

In [ ]:
# =============================================================================
# Exponential weighting: weight decay visualization
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 4))

# Show weight assigned to each past observation for different alpha values
n_past = 100
obs_idx = np.arange(n_past)  # 0 = most recent, n_past-1 = oldest

for alpha, color in [(0.05, "#3498db"), (0.10, "#e67e22"), (0.25, "#e74c3c")]:
    weights = alpha * (1 - alpha) ** obs_idx
    weights /= weights.sum()  # Normalize for comparison
    ax.plot(obs_idx, weights, label=f"α={alpha}", color=color, lw=2)

ax.set_xlabel("Steps ago (0 = most recent)")
ax.set_ylabel("Normalized weight")
ax.set_title("Exponential Recency Weighting: Weight Given to Each Past Observation\n"
             "Higher α = faster forgetting of old data")
ax.legend()
ax.set_xlim(-1, n_past)
plt.tight_layout()
plt.show()

print("Weight decay shown — higher α forgets older rewards faster. [PASS]")

---
## 10. Summary & References

### Algorithm Summary Table

| Algorithm | Exploration Strategy | Regret Rate | Computational Cost | Handles Non-Stationary? |
|-----------|---------------------|-------------|-------------------|------------------------|
| Greedy ($\varepsilon=0$) | None | $\Theta(T)$ linear | $O(K)$ | No |
| $\varepsilon$-greedy | Uniform random with prob. $\varepsilon$ | $\Theta(\varepsilon T)$ linear | $O(K)$ | No (with sample mean) |
| Exp. $\varepsilon$-greedy ($\varepsilon_t=c/t$) | Decaying random | $O(\log T)$* | $O(K)$ | No |
| UCB1 | Optimism: confidence bonus | $O(\sqrt{KT\ln T})$ | $O(K)$ | Partially (with resets) |
| Thompson Sampling | Posterior sampling | $O(\sqrt{KT\ln T})$ | $O(K)$ + sampling | No (posterior accumulates) |
| Sliding Window | Any + window | Adaptive | $O(KW)$ | Yes |
| Exp. Weighted | Any + $\alpha$-step | Adaptive | $O(K)$ | Yes |

*Requires optimal tuning of $c$.

### Key Takeaways

1. **Exploration is necessary**: Pure greedy suffers linear regret because it locks onto sub-optimal arms. Any constant $\varepsilon > 0$ guarantees all arms are eventually explored.

2. **Directed exploration beats random**: UCB1 and Thompson Sampling outperform $\varepsilon$-greedy by targeting uncertain arms rather than exploring uniformly. Their regret grows as $O(\sqrt{KT\ln T})$ vs. $\Theta(\varepsilon T)$.

3. **Thompson Sampling is Bayes-optimal**: By maintaining a full posterior, it automatically balances exploration with uncertainty and performs competitively — often the empirical winner despite similar theoretical bounds to UCB.

4. **Non-stationarity requires forgetting**: Sample-mean estimators give too much weight to stale observations. Exponential recency-weighting ($\alpha$-step) adapts to changes in $O(1)$ memory; sliding windows offer a harder memory cutoff.

5. **Gap structure determines difficulty**: The harder the problem (small $\Delta_k$), the longer it takes any algorithm to identify the optimal arm — reflected in the gap-dependent UCB regret bound $O(\ln T / \Delta_k)$.

### References

- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press. Chapter 2.
- Auer, P., Cesa-Bianchi, N., & Fischer, P. (2002). Finite-time analysis of the multiarmed bandit problem. *Machine Learning*, 47(2-3), 235–256.
- Thompson, W. R. (1933). On the likelihood that one unknown probability exceeds another in view of the evidence of two samples. *Biometrika*, 25(3-4), 285–294.
- Russo, D., Van Roy, B., Kazerouni, A., Osband, I., & Wen, Z. (2018). A tutorial on Thompson Sampling. *Foundations and Trends in Machine Learning*, 11(1), 1–96.
- Lattimore, T., & Szepesvári, C. (2020). *Bandit Algorithms*. Cambridge University Press. (Open access: https://banditalgs.com)
- Lai, T. L., & Robbins, H. (1985). Asymptotically efficient adaptive allocation rules. *Advances in Applied Mathematics*, 6(1), 4–22.